# Wanderbricks Properties

**Dataset:** `samples.wanderbricks.properties`

**Difficulty:** Easy

**Topics:** filter, aggregation, groupBy

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — Filtering and Aggregating Properties

| Function | What it does |
|----------|-------------|
| `df.filter(F.col("col").between(a, b))` | Range filter (inclusive on both ends) |
| `df.filter(F.col("col").isin(["v1", "v2"]))` | Match any value in a list |
| `df.filter((cond1) & (cond2))` | AND of two conditions (use `&`, not `and`) |
| `df.filter((cond1) \| (cond2))` | OR of two conditions (use `\|`, not `or`) |
| `F.round(col, 2)` | Round to 2 decimal places |

**Docs:** [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html) · [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.wanderbricks.properties")

# Explore the schema
df.printSchema()

# AND condition: properties with > 2 bedrooms AND base_price under 200
df.filter(
    (F.col("bedrooms") > 2) & (F.col("base_price") < 200)
).select("property_id", "property_type", "bedrooms", "base_price").show(5)

## Problem 1

Count the number of properties for each **property type** in the Wanderbricks
platform. Load `samples.wanderbricks.properties`, group by `property_type`,
and sort from most to fewest.

**Expected output columns:**
- `property_type` - type of property (e.g., Apartment, Villa, Cottage)
- `count` - number of properties of that type (sorted descending)

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = df.groupBy("property_type").count().orderBy(F.col("count").desc())

In [0]:
display(result_1)

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'property_type' in cols, "Missing column: property_type"
assert 'count' in cols, "Missing column: count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['count'] for r in result_1.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by count descending"
assert all(c > 0 for c in counts), "All count values must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Calculate the **average base price** for each property type.
Sort results so the most expensive property types appear first.

**Expected output columns:**
- `property_type` - type of property
- `avg_price` - average `base_price` for that type (sorted descending)

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = df.groupBy("property_type").agg(
    F.avg("base_price").alias("avg_price")
).orderBy(F.col("avg_price").desc())

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'property_type' in cols, "Missing column: property_type"
assert 'avg_price' in cols, "Missing column: avg_price"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
prices = [float(r['avg_price']) for r in result_2.collect()]
assert prices == sorted(prices, reverse=True), "Results must be sorted by avg_price descending"
assert all(p > 0 for p in prices), "All avg_price values must be positive"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Find all properties that have **3 or more bedrooms** (`bedrooms >= 3`).
These are larger properties suitable for families or groups.

**Expected output columns:**
- `property_id` - property identifier
- `title` - property title
- `property_type` - type of property
- `bedrooms` - number of bedrooms (must be >= 3)
- `base_price` - nightly base price

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = df.filter(F.col("bedrooms") >= 3).select(
    "property_id",
    "title",
    "property_type",
    "bedrooms",
    "base_price"
)

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'property_id' in cols, "Missing column: property_id"
assert 'title' in cols, "Missing column: title"
assert 'property_type' in cols, "Missing column: property_type"
assert 'bedrooms' in cols, "Missing column: bedrooms"
assert 'base_price' in cols, "Missing column: base_price"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_beds = result_3.agg(F.min('bedrooms')).collect()[0][0]
assert min_beds >= 3, f"All bedrooms must be >= 3, found min={min_beds}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find the **top 10 hosts** by number of properties listed on the platform.
Group by `host_id` and sort descending.

**Expected output columns:**
- `host_id` - host user identifier
- `property_count` - number of properties listed by that host (top 10)

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = df.groupBy("host_id").agg(
    F.count("*").alias("property_count")
).orderBy(F.col("property_count").desc()).limit(10)

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'host_id' in cols, "Missing column: host_id"
assert 'property_count' in cols, "Missing column: property_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt == 10, f"Expected exactly 10 rows (top 10), got {cnt}"
counts = [r['property_count'] for r in result_4.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by property_count descending"
assert all(c > 0 for c in counts), "All property_count values must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find the **10 most affordable properties** that can accommodate at least
2 guests (`max_guests >= 2`). Sort by `base_price` ascending so the
cheapest properties appear first.

**Expected output columns:**
- `property_id` - property identifier
- `title` - property title
- `property_type` - type of property
- `base_price` - nightly base price (sorted ascending)
- `max_guests` - maximum number of guests allowed (must be >= 2)

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = df.filter(F.col("max_guests") >= 2).select(
    "property_id",
    "title",
    "property_type",
    "base_price",
    "max_guests"
).orderBy(F.col("base_price")).limit(10)

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'property_id' in cols, "Missing column: property_id"
assert 'title' in cols, "Missing column: title"
assert 'property_type' in cols, "Missing column: property_type"
assert 'base_price' in cols, "Missing column: base_price"
assert 'max_guests' in cols, "Missing column: max_guests"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt == 10, f"Expected exactly 10 rows, got {cnt}"
min_guests = result_5.agg(F.min('max_guests')).collect()[0][0]
assert min_guests >= 2, f"All max_guests must be >= 2, found min={min_guests}"
prices = [float(r['base_price']) for r in result_5.collect()]
assert prices == sorted(prices), "Results must be sorted by base_price ascending"
print(f"Problem 5 passed ✓  ({cnt} rows)")